# 01. EDA & Topic Modeling Preprocessing

**Goal:** Explore the sampled dataset and apply aggressive text cleaning (Stopword removal + Lemmatization). Topic modeling is highly sensitive to noise!

In [1]:
import pandas as pd
import sys
import nltk
sys.path.append('../../../shared')
from text_utils import clean_text_basic, remove_stopwords, apply_lemmatization

## 1. Load Data

In [2]:
df = pd.read_csv('../data/raw/dataset.csv')
print(df.head())
print(f'Total headlines: {len(df)}')

   publish_date                                      headline_text
0      20030819  register recognises stations woolscouring history
1      20070827               mcgowan casts doubt over pilbara atc
2      20190822     why the congestion clogging our cities is only
3      20140818  paraplegic victim sues cedar college over fall...
4      20140729                    elders mark allison farmwriters
Total headlines: 100000


## 2. Aggressive Cleaning

> **📌 Decision Note — Why Topic Modeling Preprocessing?**
>
> **Chosen approach:** Stopword Removal + Lemmatization + Dropping short words
>
> **Why this works:** Topic models group words that co-occur frequently. Stopwords ('the', 'and') and unlemmatized variants ('running', 'runs') will dominate the topics with useless noise if not handled.
>
> **Alternatives we could have used:**
> | Option | Pros | Cons |
> |--------|------|------|
> | Basic Cleaning Only | Fastest approach | Topics will just be filled with 'to', 'of', 'in'. |
> | Stemming | Reduces vocabulary | Creates non-words (e.g. 'police' -> 'polic'), making the final extracted topics hard for humans to read. |
>
> **Why we chose this over alternatives:** For unsupervised learning where the output must be human-interpretable, Lemmatization is strongly preferred over Stemming.

In [3]:
from nltk.corpus import stopwords
import nltk
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)
stop_words_set = set(stopwords.words('english'))

def aggressive_clean(text):
    # Basic cleaning
    text = clean_text_basic(text)
    # Tokenize and remove stopwords
    from nltk.tokenize import word_tokenize
    tokens = word_tokenize(text)
    tokens = remove_stopwords(tokens, stop_words_set)
    # Keep only words with length >= 3 to filter out noise
    tokens = [t for t in tokens if len(t) > 2]
    return ' '.join(tokens)

df['cleaned'] = df['headline_text'].apply(aggressive_clean)
df['lemmatized'] = df['cleaned'].apply(apply_lemmatization)
df[['headline_text', 'lemmatized']].head(10)

,headline_text,lemmatized
0,register recognises stations woolscouring history,register recognises station woolscouring history
1,mcgowan casts doubt over pilbara atc,mcgowan cast doubt pilbara atc
2,why the congestion clogging our cities is only,congestion clogging city
3,paraplegic victim sues cedar college over fall...,paraplegic victim sue cedar college fallen tre...
4,elders mark allison farmwriters,elder mark allison farmwriters
5,mining investment reaches seven year high,mining investment reach seven year high
6,hendra may have spread to southern states,hendra may spread southern state
7,gold mine now declared safe,gold mine declared safe
8,kennedy final journey,kennedy final journey
9,push for independent nsw donations audit,push independent nsw donation audit


In [4]:
df.to_csv('../data/processed/processed_headlines.csv', index=False)
print('Saved processed data.')

Saved processed data.
